In [8]:
import pandas as pd
import pickle
import numpy as np
import talib
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
with open('../../data/nifty_data.pkl', 'rb') as file:
    nifty_data = pickle.load(file)

In [4]:
# download data for 
# S&P 500
# USDINR
# Gold
# India VIX
# ^NSEBANK
# ^CNXIT
# ^CNXFMCG

# sp500_data = yf.Ticker('^GSPC').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)
# usdinr_data = yf.Ticker('USDINR=X').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)
# gold_data = yf.Ticker('GC=F').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)  
# vix_data = yf.Ticker('^INDIAVIX').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)
# nse_bank_data = yf.Ticker('^NSEBANK').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)
# cnx_it_data = yf.Ticker('^CNXIT').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)
# cnx_fmcg_data = yf.Ticker('^CNXFMCG').history(start=nifty_data.index[0].strftime('%Y-%m-%d'), end=nifty_data.index[-1].strftime('%Y-%m-%d'), auto_adjust=True)

In [5]:
# with open('../../data/sp500_data.pkl', 'wb') as file:
#     pickle.dump(sp500_data, file)
# with open('../../data/usdinr_data.pkl', 'wb') as file:
#     pickle.dump(usdinr_data, file)
# with open('../../data/gold_data.pkl', 'wb') as file:
#     pickle.dump(gold_data, file)
# with open('../../data/vix_data.pkl', 'wb') as file:
#     pickle.dump(vix_data, file)
# with open('../../data/nse_bank_data.pkl', 'wb') as file:
#     pickle.dump(nse_bank_data, file)
# with open('../../data/cnx_it_data.pkl', 'wb') as file:
#     pickle.dump(cnx_it_data, file)

with open('../../data/sp500_data.pkl', 'rb') as file:
    sp500_data = pickle.load(file)
with open('../../data/usdinr_data.pkl', 'rb') as file:
    usdinr_data = pickle.load(file)
with open('../../data/gold_data.pkl', 'rb') as file:
    gold_data = pickle.load(file)
with open('../../data/vix_data.pkl', 'rb') as file:
    vix_data = pickle.load(file)
with open('../../data/nse_bank_data.pkl', 'rb') as file:
    nse_bank_data = pickle.load(file)
with open('../../data/cnx_it_data.pkl', 'rb') as file:
    cnx_it_data = pickle.load(file)

In [49]:
sp500_data.index = sp500_data.index.tz_localize(None)
usdinr_data.index = usdinr_data.index.tz_localize(None) 
gold_data.index = gold_data.index.tz_localize(None) 
vix_data.index = vix_data.index.tz_localize(None) 
nse_bank_data.index = nse_bank_data.index.tz_localize(None)
nifty_data.index = nifty_data.index.tz_localize(None)
cnx_it_data.index = cnx_it_data.index.tz_localize(None)

In [50]:
for data, name in zip([sp500_data, usdinr_data, gold_data, vix_data, nse_bank_data, cnx_it_data],
                      ['sp500', 'usdinr', 'gold', 'vix', 'nse_bank', 'cnx_it']):
    data = data.reindex(nifty_data.index)
    data.fillna(method='ffill', inplace=True)
    data.fillna(method='bfill', inplace=True)
    # nifty_data[f'{name}_close'] = data['Close']


/var/folders/jf/1r7jb1_s6ys1p0n0n56s2qsh0000gn/T/ipykernel_3608/3237319926.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='ffill', inplace=True)
/var/folders/jf/1r7jb1_s6ys1p0n0n56s2qsh0000gn/T/ipykernel_3608/3237319926.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='bfill', inplace=True)


In [51]:
df = nifty_data.copy()

In [52]:
df.isnull().sum()

Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

In [ ]:
df["log_return_1"]  = np.log(df["Close"] / df["Close"].shift(1))
df["log_return_20"] = np.log(df["Close"] / df["Close"].shift(20))
df["rolling_sharpe_20"] = (
    df["log_return_1"].rolling(20).mean() /
    df["log_return_1"].rolling(20).std()
)
log_ret = np.log(df["Close"] / df["Close"].shift(1))
# df["realized_vol_5"]  = np.sqrt((log_ret ** 2).rolling(5).sum())
df["realized_vol_20"] = np.sqrt((log_ret ** 2).rolling(20).sum())
df["realized_vol_60"] = np.sqrt((log_ret ** 2).rolling(60).sum())
df["vol_slope_20"] = df["realized_vol_20"].diff()
df["true_range"] = talib.TRANGE(
    df["High"].values,
    df["Low"].values,
    df["Close"].values
)
range_ratio = (df["High"] - df["Low"]) / df["Close"]
df["range_ratio_20"] = range_ratio.rolling(20).mean()

def rolling_max_drawdown(close, window):
    rolling_max = close.rolling(window).max()
    drawdown = close / rolling_max - 1.0
    return drawdown.rolling(window).min()
# df["rolling_max_dd_20"] = rolling_max_drawdown(df["Close"], 20)
df["rolling_max_dd_60"] = rolling_max_drawdown(df["Close"], 60)
def rolling_autocorr(series, window, lag=1):
    return series.rolling(window).apply(
        lambda x: x.autocorr(lag=lag),
        raw=False
    )

# df["autocorr_5"]  = rolling_autocorr(log_ret, 5)
df["autocorr_20"] = rolling_autocorr(log_ret, 20)

df["vix_slope_20"] = vix_data["Close"].diff().rolling(20).mean()

In [54]:
df["rel_spx_20"] = (
    np.log(df["Close"] / df["Close"].shift(20)) -
    np.log(sp500_data["Close"] / sp500_data["Close"].shift(20))
)
# df["rel_usdinr_20"] = (
#     np.log(df["Close"] / df["Close"].shift(20)) -
#     np.log(usdinr_data["Close"] / usdinr_data["Close"].shift(20))
# )

# df["usdinr_return_20"] = np.log(
#     usdinr_data["Close"] / usdinr_data["Close"].shift(20)
# )
df["usdinr_vol_20"] = (
    np.log(usdinr_data["Close"] / usdinr_data["Close"].shift(1))
    .rolling(20).std()
)

df["rel_gold_20"] = (
    np.log(df["Close"] / df["Close"].shift(20)) -
    np.log(gold_data["Close"] / gold_data["Close"].shift(20))
)
df["vix_level_20"] = vix_data["Close"].rolling(20).mean()
df["vix_slope_20"] = vix_data["Close"].diff().rolling(20).mean()

In [55]:
sector_df = pd.DataFrame({
    "nse_bank": nse_bank_data["Close"],
    "cnx_it": cnx_it_data["Close"]
})

In [56]:
sector_returns = np.log(sector_df / sector_df.shift(1))

In [57]:
daily_dispersion = sector_returns.std(axis=1)
daily_dispersion

Date
2007-09-17         NaN
2007-09-18    0.015977
2007-09-19    0.019617
2007-09-20    0.014390
2007-09-21    0.001249
                ...   
2025-09-12    0.000299
2025-09-15    0.005108
2025-09-16    0.002740
2025-09-17    0.000155
2025-09-18    0.002879
Length: 4146, dtype: float64

In [58]:
df["sector_dispersion_20"] = daily_dispersion.rolling(20).mean()

In [59]:
df["trend_strength_200"] = (
    df["Close"] - df["Close"].rolling(200).mean()
) / df["Close"].rolling(200).std()

In [60]:
market_vol = np.log(df["Close"] / df["Close"].shift(1)).rolling(20).std()

df["sector_dispersion_ratio_20"] = (
    daily_dispersion.rolling(20).mean() / market_vol
)

In [61]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits',
       'log_return_1', 'log_return_20', 'rolling_sharpe_20', 'realized_vol_20',
       'realized_vol_60', 'vol_slope_20', 'true_range', 'range_ratio_20',
       'rolling_max_dd_60', 'autocorr_20', 'vix_slope_20', 'rel_spx_20',
       'usdinr_vol_20', 'rel_gold_20', 'vix_level_20', 'sector_dispersion_20',
       'trend_strength_200', 'sector_dispersion_ratio_20'],
      dtype='object')

In [ ]:
NEW_FEATURES = [
    'rolling_sharpe_20', 
    'rolling_max_dd_60',

    'realized_vol_20',
    'vol_slope_20', 

    'autocorr_20',

    'rel_spx_20',
    'usdinr_vol_20', 
    'rel_gold_20', 
    'vix_level_20',

    'vix_slope_20', 
    'trend_strength_200',

    'sector_dispersion_ratio_20', 
]
df_feat = df[NEW_FEATURES].dropna()

In [67]:
temp_copy = df_feat.copy()
df_feat = df_feat.drop("Close", axis=1)

In [68]:
df_feat

,log_return_20,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,realized_vol_60,vol_slope_20,range_ratio_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20,sector_dispersion_20
Date,,,,,,,,,,,,,,,,
2008-07-07,-0.110522,-0.224338,-0.254667,0.110180,0.141355,-0.003398,0.033724,-0.219374,-0.027527,0.002786,-0.145529,32.1885,0.2430,-2.110362,0.448846,0.011056
2008-07-08,-0.109431,-0.222237,-0.254667,0.110072,0.141512,-0.000108,0.033671,-0.215630,-0.042579,0.002831,-0.139380,32.4790,0.2905,-2.162983,0.498549,0.012274
2008-07-09,-0.084490,-0.160266,-0.254667,0.116441,0.147411,0.006369,0.035011,-0.235864,0.002960,0.003098,-0.150691,32.7105,0.2315,-1.850902,0.452034,0.011915
2008-07-10,-0.086740,-0.164720,-0.254667,0.116395,0.147115,-0.000045,0.034202,-0.194999,-0.023294,0.002778,-0.153996,32.8575,0.1470,-1.824591,0.454354,0.011963
2008-07-11,-0.109400,-0.203813,-0.254667,0.119516,0.148183,0.003121,0.035888,-0.202349,-0.031528,0.003048,-0.208239,32.9730,0.1155,-1.999600,0.500986,0.013446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-12,0.019893,0.187062,-0.049712,0.023600,0.042602,-0.000214,0.006597,0.116938,0.002157,0.002392,-0.070137,11.3530,-0.1010,1.094953,1.386790,0.007374
2025-09-15,0.017622,0.164607,-0.049712,0.023662,0.042633,0.000062,0.006620,0.116114,-0.007711,0.002327,-0.081116,11.2550,-0.0980,1.037760,1.417814,0.007589
2025-09-16,0.014453,0.141348,-0.049712,0.022518,0.041226,-0.001145,0.006659,0.016878,-0.009692,0.002284,-0.087393,11.1515,-0.1035,1.215327,1.422829,0.007274


In [69]:
mean = df_feat.mean()
std = df_feat.std()

df_feat = (df_feat - mean) / (std + 1e-8)

In [70]:
df_feat

,log_return_20,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,realized_vol_60,vol_slope_20,range_ratio_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20,sector_dispersion_20
Date,,,,,,,,,,,,,,,,
2008-07-07,-1.979133,-1.233993,-1.790170,1.966213,1.088903,-0.832409,2.285305,-1.110913,-0.630254,-0.560954,-2.102724,1.329296,0.784362,-2.212219,-1.425551,0.752250
2008-07-08,-1.961051,-1.225482,-1.790170,1.962786,1.092023,-0.026427,2.279140,-1.092259,-0.957006,-0.545195,-2.014768,1.361443,0.934274,-2.254020,-1.277139,1.136946
2008-07-09,-1.547604,-0.974456,-1.790170,2.165194,1.209489,1.560028,2.433614,-1.193078,0.031579,-0.450395,-2.176565,1.387060,0.748067,-2.006110,-1.416032,1.023520
2008-07-10,-1.584896,-0.992495,-1.790170,2.163750,1.203590,-0.011146,2.340387,-0.989460,-0.538346,-0.563817,-2.223838,1.403327,0.481381,-1.985209,-1.409105,1.038556
2008-07-11,-1.960536,-1.150854,-1.790170,2.262930,1.224857,0.764398,2.534711,-1.026087,-0.717093,-0.468354,-2.999728,1.416108,0.381965,-2.124233,-1.269861,1.506830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-12,0.182761,0.432479,0.656711,-0.785516,-0.877555,-0.052374,-0.841782,0.564802,0.014138,-0.700379,-1.024336,-0.976323,-0.301320,0.334005,1.375161,-0.410826
2025-09-15,0.145118,0.341520,0.656711,-0.783531,-0.876945,0.015284,-0.839119,0.560695,-0.200074,-0.723288,-1.181373,-0.987168,-0.291852,0.288573,1.467799,-0.342798
2025-09-16,0.092581,0.247303,0.656711,-0.819908,-0.904957,-0.280392,-0.834702,0.066239,-0.243083,-0.738806,-1.271157,-0.998621,-0.309210,0.429627,1.482773,-0.442287


In [79]:
mean = df_feat.mean()
std = df_feat.std()

df_feat = (df_feat - mean) / (std + 1e-8)

In [81]:
df_feat["Close"] = temp_copy["Close"]

In [82]:
df_feat.isnull().sum()

rolling_sharpe_20             0
rolling_max_dd_60             0
realized_vol_20               0
vol_slope_20                  0
autocorr_20                   0
rel_spx_20                    0
usdinr_vol_20                 0
rel_gold_20                   0
vix_level_20                  0
vix_slope_20                  0
trend_strength_200            0
sector_dispersion_ratio_20    0
Close                         0
dtype: int64

In [83]:
with open('../../data/nifty_ts2vec.pkl', 'wb') as file:
    pickle.dump(df_feat, file)

In [14]:
from typing import Dict, Iterable, List
from matplotlib.pyplot import close
from pydantic import BaseModel, Field, ConfigDict
from typing_extensions import Literal
import numpy as np
import pandas as pd
import talib
import enum
import yfinance as yf

# create a enum for the stock names
class StockName(enum.Enum):
    NIFTY = "nifty_data"
    SP500 = "sp500_data"
    USDINR = "usdinr_data"
    GOLD = "gold_data"
    VIX = "vix_data"
    NSE_BANK = "nse_bank_data"
    CNX_IT = "cnx_it_data"


stocks = [
    {"name": "nifty_data", "ticker": "^NSEI"},
    {"name": "sp500_data", "ticker": "^GSPC"},
    {"name": "usdinr_data", "ticker": "USDINR=X"},
    {"name": "gold_data", "ticker": "GC=F"},
    {"name": "vix_data", "ticker": "^INDIAVIX"},
    {"name": "nse_bank_data", "ticker": "^NSEBANK"},
    {"name": "cnx_it_data", "ticker": "^CNXIT"},
]

class OHLCVData(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    name: Literal["nifty_data", "sp500_data", "usdinr_data", "gold_data", "vix_data", "nse_bank_data", "cnx_it_data"]
    data: pd.DataFrame

class DataObject(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    nifty_data: OHLCVData
    sp500_data: OHLCVData
    usdinr_data: OHLCVData
    gold_data: OHLCVData
    vix_data: OHLCVData
    nse_bank_data: OHLCVData
    cnx_it_data: OHLCVData

class YahooFinanceDataLoader:

    def __init__(self, stock: Dict[str, str]):
        self.stock = stock

    def load_data(self, ticker: str, start: str, end: str) -> pd.DataFrame:
        return yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True)
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.index = df.index.tz_localize(None)
        df = df[["Close", "Open", "High", "Low", "Volume"]]
        return df
    
    def run(self, start: str, end: str) -> DataObject:
        data_dict = {}
        for key, ticker in self.stock.items():
            raw_data = self.load_data(ticker, start, end)
            cleaned_data = self.clean_data(raw_data)
            data_dict[key] = OHLCVData(name=key, data=cleaned_data)
        return DataObject(**data_dict)
    
FEATURES = [
    'rolling_sharpe_20', 
    'rolling_max_dd_60',

    'realized_vol_20',
    'vol_slope_20', 

    'autocorr_20',

    'rel_spx_20',
    'usdinr_vol_20', 
    'rel_gold_20', 
    'vix_level_20',

    'vix_slope_20', 
    'trend_strength_200',

    'sector_dispersion_ratio_20', 
]

class FeatureEngineer:
    
    def __init__(self, all_data: DataObject):
        self.all_data = all_data
        self.df = self.all_data.nifty_data.data.copy()

    
    def compute_sector_dispersion(self):
        df = self.df
        nse_bank_data = self.all_data.nse_bank_data.data
        cnx_it_data = self.all_data.cnx_it_data.data

        sector_df = pd.DataFrame({
            "nse_bank": nse_bank_data["Close"],
            "cnx_it": cnx_it_data["Close"]
        })
        sector_returns = np.log(sector_df / sector_df.shift(1))
        daily_dispersion = sector_returns.std(axis=1)
        market_vol = np.log(df["Close"] / df["Close"].shift(1)).rolling(20).std()

        df["sector_dispersion_ratio_20"] = (
            daily_dispersion.rolling(20).mean() / market_vol
        )

    def compute_features(self):

        def rolling_max_drawdown(close, window):
            rolling_max = close.rolling(window).max()
            drawdown = close / rolling_max - 1.0
            return drawdown.rolling(window).min()
        
        def rolling_autocorr(series, window, lag=1):
            return series.rolling(window).apply(
                lambda x: x.autocorr(lag=lag),
                raw=False
            )
        
        df = self.df
        sp500_data = self.all_data.sp500_data.data
        usdinr_data = self.all_data.usdinr_data.data
        gold_data = self.all_data.gold_data.data
        vix_data = self.all_data.vix_data.data


        df["log_return_1"]  = np.log(df["Close"] / df["Close"].shift(1))
        log_ret = np.log(df["Close"] / df["Close"].shift(1))
        df["rolling_sharpe_20"] = (
            df["log_return_1"].rolling(20).mean() /
            df["log_return_1"].rolling(20).std()
        )
        df["rolling_max_dd_60"] = rolling_max_drawdown(df["Close"], 60)
        df["realized_vol_20"] = np.sqrt((log_ret ** 2).rolling(20).sum())
        df["vol_slope_20"] = df["realized_vol_20"].diff()
        df["autocorr_20"] = rolling_autocorr(log_ret, 20)

        df["rel_spx_20"] = (
            np.log(df["Close"] / df["Close"].shift(20)) -
            np.log(sp500_data["Close"] / sp500_data["Close"].shift(20))
        )
        df["usdinr_vol_20"] = (
            np.log(usdinr_data["Close"] / usdinr_data["Close"].shift(1))
            .rolling(20).std()
        )
        df["rel_gold_20"] = (
            np.log(df["Close"] / df["Close"].shift(20)) -
            np.log(gold_data["Close"] / gold_data["Close"].shift(20))
        )
        df["vix_level_20"] = vix_data["Close"].rolling(20).mean()
        df["vix_slope_20"] = vix_data["Close"].diff().rolling(20).mean()
        df["trend_strength_200"] = (
            df["Close"] - df["Close"].rolling(200).mean()
        ) / df["Close"].rolling(200).std()

        # df = df[FEATURES + ["Close"]].dropna()
        # self.df = df

    def post_feature_engineering_cleanup(self):
        df = self.df
        df = df[FEATURES + ["Close"]].dropna()
        self.df = df

    def standardize_features(self):
        df = self.df
        mean = df[FEATURES].mean()
        std = df[FEATURES].std()
        df[FEATURES] = (df[FEATURES] - mean) / (std + 1e-8)

    def run(self):
        self.compute_features()
        print("Features computed", self.df.columns)
        self.compute_sector_dispersion()
        print("Sector dispersion computed", self.df.columns)
        self.post_feature_engineering_cleanup()
        print("Post feature engineering cleanup done", self.df.columns)
        self.standardize_features()
        return self.df[FEATURES + ["Close"]].dropna()


# Usage


In [15]:
data_loader = YahooFinanceDataLoader(stock={stock["name"]: stock["ticker"] for stock in stocks})
all_data = data_loader.run(start="2025-01-01", end="2025-12-31")

In [16]:
feature_engineer = FeatureEngineer(all_data)
df_features = feature_engineer.run()

Features computed Index(['Close', 'Open', 'High', 'Low', 'Volume', 'log_return_1',
       'rolling_sharpe_20', 'rolling_max_dd_60', 'realized_vol_20',
       'vol_slope_20', 'autocorr_20', 'rel_spx_20', 'usdinr_vol_20',
       'rel_gold_20', 'vix_level_20', 'vix_slope_20', 'trend_strength_200'],
      dtype='object')
Sector dispersion computed Index(['Close', 'Open', 'High', 'Low', 'Volume', 'log_return_1',
       'rolling_sharpe_20', 'rolling_max_dd_60', 'realized_vol_20',
       'vol_slope_20', 'autocorr_20', 'rel_spx_20', 'usdinr_vol_20',
       'rel_gold_20', 'vix_level_20', 'vix_slope_20', 'trend_strength_200',
       'sector_dispersion_ratio_20'],
      dtype='object')
Post feature engineering cleanup done Index(['rolling_sharpe_20', 'rolling_max_dd_60', 'realized_vol_20',
       'vol_slope_20', 'autocorr_20', 'rel_spx_20', 'usdinr_vol_20',
       'rel_gold_20', 'vix_level_20', 'vix_slope_20', 'trend_strength_200',
       'sector_dispersion_ratio_20', 'Close'],
      dtype='objec

In [17]:
df_features

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20,Close
Date,,,,,,,,,,,,,
2025-10-20,0.189940,-1.067188,1.878769,0.314399,2.428021,0.511097,0.081952,-2.415461,-1.231261,0.866258,1.443930,0.255842,25843.150391
2025-10-21,0.539634,-1.067188,1.542110,-0.396571,2.274223,0.529070,0.007747,-0.680635,-1.172329,0.510309,1.467377,-0.461972,25868.599609
2025-10-23,0.665448,-1.067188,1.528660,0.051926,2.190078,0.230552,-0.459688,-1.108554,-1.084728,0.707450,1.481708,-0.098621,25891.400391
2025-10-24,0.718418,-1.067188,1.437748,-0.055563,1.736774,0.170924,-0.446006,-0.816969,-0.999516,0.691022,0.897686,0.076298,25795.150391
2025-10-27,1.544395,-1.067188,1.428914,0.058332,0.071893,0.349160,-0.430530,0.421267,-0.913508,0.696498,1.652873,0.087790,25966.050781
2025-10-28,2.502070,-1.067188,0.062372,-1.825681,-0.993894,0.843623,-0.038761,0.938509,-0.872096,0.389833,1.395461,0.814908,25936.199219
2025-10-29,2.900579,-1.067188,0.375702,0.505381,-1.421147,1.269564,-0.092986,1.118808,-0.824314,0.433643,1.867015,0.975169,26053.900391
2025-10-30,2.191720,-1.067188,1.064060,1.025786,-1.210163,1.500720,-0.086649,0.718658,-0.744676,0.652689,0.895162,0.227814,25877.849609
2025-10-31,1.230185,-1.067188,0.352395,-0.916946,0.102362,0.641224,0.095578,0.725230,-0.596551,1.123638,0.045556,-0.067077,25722.099609
